In [ ]:
from util.unet import UNet
class UNetWrapper(nn.Module):
    def __init__(self, **kwargs):
        super().__init__()
        
        self.input_batchnorm = nn.BatchNorm2d(kwargs['in_channels'])
        self.unet = UNet(**kwargs)
        self.final = nn.Sigmoid()
        
        self._init_weights()
        
    def forward(self, input_batch):
        bn_output = self.input_batchnorm(input_batch)
        un_output = self.unet(bn_output)
        fn_output = self.final(un_output)
        return fn_output

#第一个问题是关于输入输出图像的尺寸。
#根据论文中的情况，UNet网络接收一个572×572大小的图像，输出一个388×388大小的图像，
#而我们期望输入和输出能够一样大，毕竟我们是在做一个医学项目，
#边角信息的丢失也可能会导致判断失误。
#正好在UNet网络中开启padding就可以解决这个问题。

##第二个问题是我们的数据是三维数据，是512×512×128的图像，
##如果直接塞进UNet我们的内存就炸了。我算了一下，这一个图像就是128MB，
##UNet的第一层有64个channel，那我们就需要128×64MB，也就是8GB的空间。
##考虑之前二维图像的RGB三个通道，这里我们把不同的切片也看做通道，
##只保留正在处理的切片上下相邻的几个切片数据以通道的形式传入模型。
##当然这里会有一些信息的损失，
##因为本来这些切片之间是有上下顺序的，按二维图像的通道输入就没有了这种关系。

###第三个问题是原数据不匹配。
###前几节里面介绍的标注数据，
###给出了中心点坐标以及直径尺寸。
###但是我们需要的是一个图像区域，
###来标明里面的哪些像素块是结节。
###像下面画的，我们期望要这样一个效果。

### 0000000000
### 0000000000
### 0011100000
### 0011100000
### 0011000000
### 0000000000

In [ ]:
center_irc = xyz2irc(
    candidateInfo_tup.center_xyz,
    self.origin_xyz,
    self.vxSize_xyz,
    self.direction_a,
)
ci = int(center_irc.index)
cr = int(center_irc.row)
cc = int(center_irc.col)

index_radius = 2
try:
    while self.hu_a[ci + index_radius, cr, cc] > threshold_hu and self.hu_a[ci - index_radius, cr, cc] > threshold_hu:
        index_radius+= 1
except IndexError:
    index_radius -= 1

In [ ]:
def buildAnnotationMask(self, positiveInfo_list, threshold_hu = -700):
    boundingBox_a = np.zeros_like(self.hu_a, dtype=bool)
    
    for candidateInfo_tup in positiveInfo_list:
        boundingBox_a[
            ci - index_radius: ci + index_radius + 1,
            cr - row_radius: cr + row_radius + 1,
            cc - col_radius: cc + col_radius + 1
         ]  = True
        
    mask_a = boundingBox_a & (self.hu_a > threshould_hu)
    
    return mask_a

In [ ]:
class Ct:
    def __init__(self, series_uid):
        candidateInfo_list = getCandidateInfoDict()[self.series_uid] #获取候选信息

        self.positiveInfo_list = [  #正样本列表
            candidate_tup
            for candidate_tup in candidateInfo_list
            if candidate_tup.isNodule_bool
        ]
        self.positive_mask = self.buildAnnotationMask(self.positiveInfo_list)  #正样本掩码构建
#最后这行是把具有非0计数的掩码切片的索引存下来
        self.positive_indexes = (self.positive_mask.sum(axis=(1,2))
                                 .nonzero()[0].tolist())    

In [ ]:
def getRawCandidate(self, center_xyz, width_irc):
#获取中心位置
        center_irc = xyz2irc(center_xyz, self.origin_xyz, self.vxSize_xyz,
                             self.direction_a)
#起始位置和终止位置索引
        slice_list = []
        for axis, center_val in enumerate(center_irc):
            start_ndx = int(round(center_val - width_irc[axis]/2))
            end_ndx = int(start_ndx + width_irc[axis])
#断言 用来处理异常
            assert center_val >= 0 and center_val < self.hu_a.shape[axis], repr([self.series_uid, center_xyz, self.origin_xyz, self.vxSize_xyz, center_irc, axis])

            if start_ndx < 0:
                # log.warning("Crop outside of CT array: {} {}, center:{} shape:{} width:{}".format(
                #     self.series_uid, center_xyz, center_irc, self.hu_a.shape, width_irc))
                start_ndx = 0
                end_ndx = int(width_irc[axis])

            if end_ndx > self.hu_a.shape[axis]:
                # log.warning("Crop outside of CT array: {} {}, center:{} shape:{} width:{}".format(
                #     self.series_uid, center_xyz, center_irc, self.hu_a.shape, width_irc))
                end_ndx = self.hu_a.shape[axis]
                start_ndx = int(self.hu_a.shape[axis] - width_irc[axis])
#加入切片信息
            slice_list.append(slice(start_ndx, end_ndx))

        ct_chunk = self.hu_a[tuple(slice_list)]
#提取其中的正样本掩码数据
        pos_chunk = self.positive_mask[tuple(slice_list)]
#返回数据
        return ct_chunk, pos_chunk, center_irc

In [ ]:
@raw_cache.memoize(typed=True)
def getCtRawCandidate(series_uid, center_xyz, width_irc):
    ct = getCt(series_uid)
    ct_chunk, pos_chunk, center_irc = ct.getRawCandidate(center_xyz,
                                                         width_irc)
    ct_chunk.clip(-1000, 1000, ct_chunk)
    return ct_chunk, pos_chunk, center_irc

In [ ]:
candidateInfo_list = []
    with open('annotations_with_malignancy.csv', "r") as f:
        for row in list(csv.reader(f))[1:]:
            series_uid = row[0]
            annotationCenter_xyz = tuple([float(x) for x in row[1:4]])
            annotationDiameter_mm = float(row[4])
            isMal_bool = {'False': False, 'True': True}[row[5]]
#从中取出我们需要的数据并合成结果
            candidateInfo_list.append(
                CandidateInfoTuple(
                    True,
                    True,
                    isMal_bool,
                    annotationDiameter_mm,
                    series_uid,
                    annotationCenter_xyz,
                )
            )
#然后从candidates.csv取出候选信息
    with open('candidates.csv', "r") as f:
        for row in list(csv.reader(f))[1:]:
            series_uid = row[0]

            if series_uid not in presentOnDisk_set and requireOnDisk_bool:
                continue

            isNodule_bool = bool(int(row[4]))
            candidateCenter_xyz = tuple([float(x) for x in row[1:4]])
#这里只使用非结节数据
            if not isNodule_bool:
                candidateInfo_list.append(
                    CandidateInfoTuple(
                        False,#是否结节
                        False,#是否恶性
                        False,#是否有标注
                        0.0,
                        series_uid,
                        candidateCenter_xyz,
                    )
                )

    candidateInfo_list.sort(reverse=True)
    return candidateInfo_list

In [ ]:
class Luna2dSegementationDataset(Dataset):
    def __init__(self,
                  val_stride=0,
                  isValSet_bool=None,
                series_uid=None,
                contextSlices_count=3,
                fullCt_bool=False,):
        
        self.contextSlices_count = contextSlices_count
        self.fullCt_bool = fullCt_bool
        
        if series_uid:
            self.series_list = [series_uid]
        else:
            self.series_list = sorted(getCandidateInfoDict().keys())
        
        if isValSet_bool:
            assert val_stride > 0, val_stride
            self.series_list = self.series_list[::val_stride]
            assert self.series_list
        elif val_stride > 0:
            del self.series_list[::val_stride]
            assert self.series_list
        
        self.sample_list = []
        for series_uid in self.series_list:
            index_count, positive_indexes = getCtSampleSize(series_uid)
            
            if self.fullCt_bool:
                self.sample_list += [(series_uid, slice_ndx) for slice_ndx in range(index_count)]
            else:
                self.sample_list += [(series_uid, slice_ndx) for slice_ndx in positive_indexes]
                
        self.candidateInfo_list = getCandidateInfoList()
        
        series_set = set(self.series_list)
        self.candidateInfo_list = [cit for cit in self.candidateInfo_list if cit.series_uid in series_set]
        
        self.pos_list = [nt for nt in self.candidateInfo_list if nt.isNodule_bool]
        
        log.info("{!r}: {} {} series, {} slices, {} nodules".format(
            self,
            len(self.series_list),
            {None: 'general', True:'validation', False:'training'}[isValSet_bool],
            len(self.sample_list),
            len(self.pos_list)
        ))

In [ ]:
@raw_cache.memoize(typed=True)
def getCtSampleSize(series_uid):
    ct = Ct(series_uid)
    return int(ct.hu_a.shape[0], ct.positive_indexes)

In [ ]:
def __getitem__(self, ndx):
    series_uid, slice_ndx = self.sample_list[ndx % len(self.sample_list)]
    return self.getitem_fullSlice(series_uid, slice_ndx)

def getitem_fullSice(self, series_uid, slice_ndx):
    ct = getCt(series_uid)
    ct_t = torch.zeros((self.contextSlices_count * 2 + 1, 512, 512))
    start_ndx = slice_ndx - self.contextSlices_count
    end_ndx = slice_ndx + self.contextSlices_count + 1
    
    for i, context_ndx in enumerate(range(start_ndx, end_ndx)):
        context_ndx = max(context_ndx, 0)
        context_ndx = min(context_ndx, ct.hu_a.shape[0] - 1)
        ct_t[i] = torch.form.numpy(ct.hu_a[context_ndx].astype(np.float32))
    ct_t.clamp_(-1000, 1000)
    pos_t = torch.from_numpy(ct.positive_mask[slice_ndx]).unsqueeze(0)
    
    return ct_t, pos_t, ct.series_uid, slice_ndx

In [ ]:
class TrainingLuna2dSegmentationDataset(Luna2dSegmentationDataset):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ratio_int = 2
    
    def __len__(self):
        return 200000
    
    def shuffleSamples(self):
        random.shuffle(self.candidateInfo_list)
        random.shuffe(self.pos_list)
        
    def __getitem__(self, ndx):
        candidateInfo_tup = self.pos_list[ndx % len(self.pos_list)]
        return self.getitem_trainingCrop(candidateInfo_tup)
    
    def getitem_trainingCrop(self, candidateInfo_tup):
        ct_a, pos_a, center_irc = getCtRawCandidate(
            candidateInfo_tup.series_uid,
        candidateInfo_tup.center_xyz,
            (7, 96, 96),
        )
        pos_a = pos_a[3:4]
        row_offset = random.randrange(0, 32)
        col_offset = random.randrange(0, 32)
        ct_t = torch.from_numpy(ct_a[:, row_offset:row_offset+64, col_offset:col_offset+64]).to(torch.float32)
        pos_t = torch.from_numpy(pos_a[:, row_offset:row_offset+64, col_offset:col_offset+64]).to(torch.long)
        
        slice_ndx = center_irc.index
        return ct_t, pos_t, candidateInfo_tup.series_uid, slice_ndx

In [ ]:
class SegmentationAugmentation(nn.Module):
    def __init__(
            self, flip=None, offset=None, scale=None, rotate=None, noise=None
    ):
        super().__init__()

        self.flip = flip
        self.offset = offset
        self.scale = scale
        self.rotate = rotate
        self.noise = noise

    def forward(self, input_g, label_g):
#这里是获取变换方法
        transform_t = self._build2dTransformMatrix()
        transform_t = transform_t.expand(input_g.shape[0], -1, -1)
#因为GPU适合处理浮点数，这里传入GPU的同时转换成浮点数
        transform_t = transform_t.to(input_g.device, torch.float32)
#affine_grid和grid_sample就是实现变换和重新采样（生成新图像）的方法
        affine_t = F.affine_grid(transform_t[:,:2],
                input_g.size(), align_corners=False)

        augmented_input_g = F.grid_sample(input_g,
                affine_t, padding_mode='border',
                align_corners=False)
#这里同时在掩码操作
        augmented_label_g = F.grid_sample(label_g.to(torch.float32),
                affine_t, padding_mode='border',
                align_corners=False)
#最后是增加噪声
        if self.noise:
            noise_t = torch.randn_like(augmented_input_g)
            noise_t *= self.noise

            augmented_input_g += noise_t

        return augmented_input_g, augmented_label_g > 0.5

    def _build2dTransformMatrix(self):
        transform_t = torch.eye(3)

        for i in range(2):
            if self.flip:
                if random.random() > 0.5:
                    transform_t[i,i] *= -1

            if self.offset:
                offset_float = self.offset
                random_float = (random.random() * 2 - 1)
                transform_t[2,i] = offset_float * random_float

            if self.scale:
                scale_float = self.scale
                random_float = (random.random() * 2 - 1)
                transform_t[i,i] *= 1.0 + scale_float * random_float

        if self.rotate:
            angle_rad = random.random() * math.pi * 2
            s = math.sin(angle_rad)
            c = math.cos(angle_rad)

            rotation_t = torch.tensor([
                [c, -s, 0],
                [s, c, 0],
                [0, 0, 1]])

            transform_t @= rotation_t

        return transform_t

# Adam优化器和Dice损失
* 梯度下降优化的发展历程：SGD->SGDM->NAG->AdaGrad->AdaDelta->Adam->Nadam
* Momentum为添加了动量后的方法，提高了收敛速度
* Nesterov方法在进行更新前先进行预演，从而找到一个更合适的梯度方向和幅度
* AdaGrad让不同的参数拥有不同的学习率，并引入梯度的平方和作为衰减项，可自动降低学习率
* AdaDelta在AdaGrad基础上进行了改进，让学习率和训练周期更匹配

## Adam优化器
* Adam = Momentum + Adaptive Learning Rate
* 既然不用的参数可以有不同的学习率，那么不同的参数是不是也可以有不同的Momentum呢?
* 基于上述想法，Adam对于每个参数，不仅仅有一个独立的学习率，还有自己的Momentum，这样在训练过程中，每个参数的更新都更加独立，提升了模型训练速度和训练的稳定性

    * 优点
    1. 计算高效
    2. 内存消耗相对较低
    3. 使用友好，很少需要调整参数
    4. 适合解决大规模数据和参数优化问题

In [ ]:
def initModel(self):
    segmentation_model = UNetWrapper(
        in_channels=7,
        n_classes=1,
        depth=3,
        wf=4,
        padding=True,
        batch_norm=True,
        up_mode='upconv',
    )
    
    augmentation_model = SegmentationAugmentation(**self.augmentation_dict)
    
    if self.use_cuda:
        log.info("Using CUDA; {} devices.".format(torch.cuda.device_count()))
        if torch.cuda.device_count() > 1:
            segmentation_model = nn.DataParallel(segmentation_model)
            augmentation_model = nn.DataParallel(augmentation_model)
        segmentation_model = segmentation_model.to(self.device)
        augmentation_model = augmentation_model.to(self.device)
        
    retrun segmentation_model, augmentation_model

In [ ]:
def initOptimizer(self):
    return Adam(self.segmentation_model.parameters())

In [ ]:
def diceLoss(self, prediction_g, label_g, epsilon=1):
    diceLabel_g = label_g.sum(dim=[1,2,3])
    dicePrediction_g = prediction_g.sum(dim=[1,2,3])
    diceCorrect_g = (prediction_g * label_g).sum(dim=[1,2,3])
    
    diceRatio_g = (2 * diceCorrect_g + epsilon) / (dicePrediction_g + diceLabel_g + epsilon)
    
    return 1 - diceRatio_g

In [ ]:
def computeBatchLoss(self, batch_ndx, batch_tup, batch_size, metrics_g, classificationThreshold=0.5):
    input_t, label_t, series_list, _slice_ndx_list = batch_tup
    
    input_g = input_t.to(self.device, non_blocking=True)
    label_t = label_t.to(self.device, non_blocking=True)
    
    if self.segmentation_model.training and self.augementation_dict:
        input_g, label_g = self. augmentation_model(input_g, label_g)
        
    prediction_g = self.segmentation_model(input_g)
    
    diceLoss_g = self.diceLoss(prediction_g, label_g)
    fnLoss_g = self.diceLoss(prediction_g * label_g, label_g)
    
    start_ndx = batch_ndx * batch_size
    end_ndx = start_ndx + input_t.size(0)
    
    with torch.no_grad():
        predictionBool_g = (prediction_g[:, 0:1] > classificationThreshold).to(torch.float32)
        
            tp = (     predictionBool_g *  label_g).sum(dim=[1,2,3])
            fn = ((1 - predictionBool_g) *  label_g).sum(dim=[1,2,3])
            fp = (     predictionBool_g * (~label_g)).sum(dim=[1,2,3])

            metrics_g[METRICS_LOSS_NDX, start_ndx:end_ndx] = diceLoss_g
            metrics_g[METRICS_TP_NDX, start_ndx:end_ndx] = tp
            metrics_g[METRICS_FN_NDX, start_ndx:end_ndx] = fn
            metrics_g[METRICS_FP_NDX, start_ndx:end_ndx] = fp
    return diceLoss_g.mean() + fnLoss_g.mean() * 8